<img src="https://raw.githubusercontent.com/Delarbol/semillero/refs/heads/master/Images-DS/alt_banner.png" width=100%>
<h1 align=center style="font-size:3vw">Departamento de Matemáticas</h1>
<h1 align=center style="font-size:3vw">IA para Economía II</h1>
<h2 align=center>Economía</h2>
<h2 align=center>Camilo Andrés De la Cruz Arboleda - David Franco</h2>



<img src="https://raw.githubusercontent.com/Delarbol/semillero/refs/heads/master/Images-DS/attrib.png">


------------------



# 🌲➡️🌳🌳🌳 De un Árbol a un Bosque: Modelos de Ensamble para la Gestión Pública  
## Caso aplicado: Predicción de evasión de impuestos en personas jurídicas

**Rol del analista:** Economista con conocimientos en Ciencia de Datos   
**Contexto:** El Ministerio de Hacienda quiere priorizar auditorías tributarias para identificar personas jurídicas con mayor probabilidad de evasión.  
**Objetivo técnico:** comparar dos modelos de ensamble — **Random Forest** y **XGBoost** — en un problema de clasificación binaria.  
**Objetivo económico:** traducir el desempeño del modelo en decisiones de negocio público: a quién auditar primero, cuánto recaudo adicional puede obtenerse y qué riesgos de sesgo o deterioro del modelo deben vigilarse.

---

## ¿Por qué este tema importa para economía pública?

En econometría tradicional solemos preguntar: **¿qué variable tiene un efecto estadísticamente significativo?**  
En analítica predictiva para política pública la pregunta cambia ligeramente: **¿a quién conviene inspeccionar primero si el presupuesto de auditoría es limitado?**

Eso no reemplaza la inferencia causal. La complementa.

En escenarios como evasión fiscal, fraude, focalización o abandono escolar, el problema operativo suele ser:

- hay **muchas observaciones**,
- el Estado tiene **capacidad limitada**,
- y se necesita un **ranking de riesgo** más que una ecuación elegantemente identificada.

Los modelos de ensamble son útiles porque:

1. **capturan no linealidades e interacciones**,
2. **reducen sobreajuste** mejor que un árbol aislado,
3. permiten construir **scores de riesgo** accionables,
4. y ofrecen herramientas modernas de interpretación como **SHAP**.

---

## Hoja de ruta

1. Simular un dataset realista de 2.000 empresas.  
2. Hacer *feature engineering* con criterio económico.  
3. Preprocesar variables categóricas con **One-Hot Encoding**.  
4. Entrenar y comparar:
   - **Random Forest**
   - **XGBoost**
5. Evaluar con:
   - Matriz de Confusión
   - Reporte de Clasificación
   - Curva de ganancia / lift
6. Interpretar variables con **SHAP**.
7. Cerrar con 3 retos:
   - tuning,
   - sesgo hacia PyMEs,
   - y *data drift* por recesión.

> **Nota pedagógica:** cada bloque de código está precedido por una explicación del **por qué técnico** y del **por qué económico**.



# 1. Librerías y configuración

Antes de modelar necesitamos un entorno reproducible.

## ¿Por qué estas librerías?

- **numpy / pandas**: simulación y manipulación de datos.
- **matplotlib / seaborn**: visualización exploratoria y de resultados.
- **scikit-learn**: partición de datos, *pipeline*, métricas y Random Forest.
- **xgboost**: implementación eficiente de Gradient Boosting con interfaz tipo scikit-learn.
- **shap**: interpretación moderna de modelos complejos.
- **ColumnTransformer + Pipeline**: cruciales para evitar fugas de información y mantener un flujo limpio y profesional.

Desde la perspectiva de MLOps, esta estructura es importante porque permite que el mismo pipeline de entrenamiento pueda luego llevarse a producción con menos fricción.


In [ ]:

# Librerías base para simulación y manipulación
import numpy as np
import pandas as pd

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de scikit-learn para modelado y evaluación
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc
)
from sklearn.ensemble import RandomForestClassifier

# Modelo de boosting
from xgboost import XGBClassifier

# Interpretabilidad
import shap

# Ajustes visuales globales
sns.set_theme(style="whitegrid")  # Estilo consistente para gráficas
plt.rcParams["figure.figsize"] = (9, 5)  # Tamaño base de gráficas
np.random.seed(42)  # Semilla para reproducibilidad



# 2. Simulación del dataset

En aplicaciones públicas no siempre podemos compartir datos reales por razones legales, tributarias o de confidencialidad.  
Por eso aquí construiremos un **dataset sintético**, pero con lógica económica plausible.

## Variables

Simularemos 2.000 empresas con:

- **ingresos_brutos**: tamaño económico de la empresa.
- **sector_economico**: sector en el que opera.
- **numero_empleados**: proxy de tamaño organizacional.
- **historial_multas**: antecedente de incumplimiento.
- **años_existencia**: antigüedad.
- **exenciones_previas**: intensidad de beneficios tributarios previos.
- **evade_impuestos**: variable objetivo binaria.

## Lógica económica de la simulación

No queremos datos aleatorios sin estructura. Queremos una “verdad generadora” razonable:

- empresas con más **multas históricas** deberían tener más riesgo de evasión;
- ciertos sectores podrían tener mayor complejidad contable y, por tanto, mayor riesgo;
- la relación entre tamaño y evasión puede ser **no lineal**;
- algunas empresas con muchas **exenciones previas** podrían mostrar patrones agresivos de planeación fiscal;
- empresas muy jóvenes o muy pequeñas pueden tener mayor fragilidad administrativa.

Esto hace que el ejercicio sea útil para entender por qué los ensambles suelen ganar frente a modelos lineales simples.


In [ ]:

# Número de empresas a simular
n = 2000

# Sectores económicos plausibles en una administración tributaria
sectores = ["Comercio", "Servicios", "Manufactura", "Construcción", "Tecnología", "Agro"]

# Simulación del sector con probabilidades no uniformes
sector_economico = np.random.choice(
    sectores,
    size=n,
    p=[0.24, 0.28, 0.15, 0.12, 0.11, 0.10]
)

# Antigüedad de la empresa (años)
anios_existencia = np.random.randint(1, 41, size=n)

# Número de empleados con asimetría positiva
numero_empleados = np.random.lognormal(mean=3.3, sigma=0.7, size=n).astype(int)

# Ingresos brutos anuales; relación parcial con tamaño
ingresos_brutos = (
    numero_empleados * np.random.uniform(20_000, 80_000, size=n)
    + np.random.lognormal(mean=12.2, sigma=0.8, size=n)
)

# Historial de multas: conteo de sanciones previas
historial_multas = np.random.poisson(lam=1.2, size=n)

# Exenciones previas: monto agregado de beneficios tributarios
exenciones_previas = np.random.gamma(shape=2.0, scale=80_000, size=n)

# Indicador PyME aproximado, útil luego para evaluación de sesgo
es_pyme = (numero_empleados < 50).astype(int)

# Efectos sectoriales sobre la probabilidad de evasión
efecto_sector = {
    "Comercio": 0.20,
    "Servicios": 0.05,
    "Manufactura": 0.10,
    "Construcción": 0.35,
    "Tecnología": -0.05,
    "Agro": 0.15
}

# Convertimos el sector a efecto numérico
sector_riesgo = np.array([efecto_sector[s] for s in sector_economico])

# Construcción de un score latente con componentes lineales y no lineales
# - multas incrementan fuertemente el riesgo
# - exenciones altas pueden asociarse con planeación agresiva
# - firmas muy pequeñas y muy jóvenes pueden tener mayor desorden administrativo
# - se incluye algo de no linealidad con log(ingresos)
score = (
    -2.3
    + 0.55 * historial_multas
    + 0.25 * np.log1p(exenciones_previas / 100_000)
    + 0.18 * sector_riesgo
    - 0.10 * np.log1p(anios_existencia)
    - 0.20 * np.log1p(numero_empleados)
    + 0.16 * np.log1p(ingresos_brutos / 1_000_000)
)

# Componente adicional: PyMEs con multas tienen riesgo más alto
score += 0.35 * ((es_pyme == 1) & (historial_multas >= 2))

# Componente adicional: empresas con muchas exenciones y pocos años de existencia
score += 0.30 * ((exenciones_previas > np.quantile(exenciones_previas, 0.75)) & (anios_existencia < 5))

# Convertimos score a probabilidad mediante función logística
prob_evasion = 1 / (1 + np.exp(-score))

# Generamos la variable binaria final
evade_impuestos = np.random.binomial(1, prob_evasion, size=n)

# Construimos el DataFrame principal
df = pd.DataFrame({
    "ingresos_brutos": ingresos_brutos,
    "sector_economico": sector_economico,
    "numero_empleados": numero_empleados,
    "historial_multas": historial_multas,
    "años_existencia": anios_existencia,
    "exenciones_previas": exenciones_previas,
    "es_pyme": es_pyme,
    "evade_impuestos": evade_impuestos
})

# Vista rápida
df.head()



## Inspección inicial

Antes de cualquier modelo, un economista serio inspecciona la base:

- ¿Qué tamaño tiene?
- ¿Hay clases desbalanceadas?
- ¿Los órdenes de magnitud tienen sentido?
- ¿Hay valores extraños?

Esto no es “cosmética”; es control de calidad analítica.


In [ ]:

# Dimensiones de la base
print("Dimensiones:", df.shape)

# Tipos de datos
print("\nTipos de datos:")
print(df.dtypes)

# Proporción de evasión
print("\nDistribución del target:")
print(df["evade_impuestos"].value_counts(normalize=True).rename("proporción"))



# 3. Feature Engineering

Aquí pasamos de “variables disponibles” a “variables útiles”.

## ¿Por qué hacer feature engineering en economía pública?

Porque muchas veces las variables que importan no son las crudas, sino los **ratios, intensidades o señales compuestas**.

Ejemplo:
- no es lo mismo tener 2 multas si la empresa factura 50 millones que si factura 50 mil millones;
- no es lo mismo tener 200 empleados con pocos ingresos por empleado que con alta productividad;
- las exenciones deben leerse en proporción al tamaño económico.

Por eso construiremos variables derivadas que capturen mejor el comportamiento tributario.


In [ ]:

# Copiamos la base para mantener trazabilidad
df_fe = df.copy()

# Ratio de multas sobre ingresos
# Se suma 1 al denominador en millones para evitar divisiones por cero o valores explosivos
df_fe["ratio_multas_ingresos"] = df_fe["historial_multas"] / (df_fe["ingresos_brutos"] / 1_000_000 + 1)

# Exenciones relativas al tamaño económico
df_fe["ratio_exenciones_ingresos"] = df_fe["exenciones_previas"] / (df_fe["ingresos_brutos"] + 1)

# Ingreso por empleado como proxy de productividad o intensidad de capital
df_fe["ingreso_por_empleado"] = df_fe["ingresos_brutos"] / (df_fe["numero_empleados"] + 1)

# Indicador de empresa "joven"
df_fe["empresa_joven"] = (df_fe["años_existencia"] <= 5).astype(int)

# Vista rápida de nuevas variables
df_fe.head()



# 4. Análisis Exploratorio (EDA)

El objetivo aquí no es hacer un libro de EDA, sino identificar señales económicas tempranas.

## Preguntas orientadoras

1. ¿Las empresas evasoras muestran más multas históricas?
2. ¿Hay sectores con mayor incidencia?
3. ¿Las exenciones parecen asociarse con mayor o menor riesgo?
4. ¿Qué patrones justifican usar modelos no lineales?

Si la realidad tributaria fuera completamente lineal, una regresión simple bastaría.  
Pero normalmente encontramos umbrales, interacciones y heterogeneidad sectorial. Ahí es donde los ensambles brillan.


In [ ]:

# Resumen descriptivo básico
df_fe.describe(include="all").T


In [ ]:

# Tasa de evasión por sector
sector_summary = (
    df_fe.groupby("sector_economico")["evade_impuestos"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

sns.barplot(data=sector_summary, x="sector_economico", y="evade_impuestos")
plt.title("Tasa media de evasión por sector")
plt.ylabel("Proporción de evasión")
plt.xlabel("Sector económico")
plt.xticks(rotation=30)
plt.show()


In [ ]:

# Distribución de ingresos por estado de evasión
sns.boxplot(data=df_fe, x="evade_impuestos", y="ingresos_brutos")
plt.title("Ingresos brutos según evasión")
plt.xlabel("Evade impuestos")
plt.ylabel("Ingresos brutos")
plt.show()


In [ ]:

# Historial de multas por evasión
sns.boxplot(data=df_fe, x="evade_impuestos", y="historial_multas")
plt.title("Historial de multas según evasión")
plt.xlabel("Evade impuestos")
plt.ylabel("Número de multas")
plt.show()



## Correlación

La matriz de correlación sirve para una primera lectura de asociación entre variables numéricas.

**Advertencia econométrica importante:** correlación no implica causalidad.  
Aquí la usamos como herramienta de exploración, no como prueba de mecanismo causal.


In [ ]:

# Selección de variables numéricas para correlación
num_cols = df_fe.select_dtypes(include=[np.number]).columns.tolist()

# Cálculo de correlación
corr = df_fe[num_cols].corr()

# Heatmap
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlación")
plt.show()



# 5. Preprocesamiento

Los modelos de árboles de scikit-learn y XGBoost requieren insumos numéricos.

## ¿Por qué usar One-Hot Encoding?

La variable `sector_economico` es categórica. No podemos pasarla directamente como texto.  
Con **One-Hot Encoding** creamos columnas binarias por categoría.

Desde el punto de vista económico esto evita imponer un orden artificial entre sectores.  
No tendría sentido codificar:

- Comercio = 1
- Servicios = 2
- Agro = 3

porque eso sugeriría una jerarquía numérica inexistente.

## ¿Por qué usar un Pipeline?

Porque el preprocesamiento debe aplicarse **exactamente igual** en entrenamiento y prueba.  
Eso reduce errores y facilita transición a producción.


In [ ]:

# Definimos variable objetivo
y = df_fe["evade_impuestos"]

# Definimos matriz de características
X = df_fe.drop(columns=["evade_impuestos"])

# Columnas categóricas
categorical_features = ["sector_economico"]

# Columnas numéricas: todas las demás
numeric_features = [col for col in X.columns if col not in categorical_features]

# Transformador para variables categóricas
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combinamos transformaciones
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# Partición train-test estratificada para conservar proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)



# 6. Modelo 1 — Random Forest

## Intuición económica y técnica

Un **árbol de decisión** individual es fácil de interpretar, pero suele tener alta varianza:  
si cambiamos ligeramente la muestra, el árbol cambia mucho.

**Random Forest** corrige eso con **Bootstrap Aggregation (Bagging)**:

1. toma muchas muestras *bootstrap* del conjunto original,
2. entrena un árbol en cada una,
3. y luego **promedia** sus decisiones.

### Analogía: el “sabio de la multitud”

En vez de confiar en un solo auditor o analista, preguntamos a muchos.  
Cada uno puede equivocarse un poco, pero el promedio de muchas opiniones reduce la variabilidad idiosincrática.

En términos estadísticos:
- un árbol individual = **bajo sesgo, alta varianza**,
- el bosque = **menor varianza** por agregación.

Esto es especialmente útil en problemas donde la estructura es compleja y no lineal, como evasión tributaria.


In [ ]:

# Definimos un pipeline completo con preprocesamiento + modelo
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,      # Número de árboles en el bosque
        max_depth=8,           # Profundidad máxima para controlar complejidad
        min_samples_leaf=10,   # Tamaño mínimo de hoja para suavizar el árbol
        random_state=42,
        class_weight="balanced"  # Ayuda si hay desbalance de clases
    ))
])

# Entrenamos el modelo
rf_model.fit(X_train, y_train)



# 7. Modelo 2 — XGBoost

## Intuición económica y técnica

Si Random Forest se parece a una votación simultánea, **Gradient Boosting** se parece a un proceso iterativo de aprendizaje institucional.

### ¿Qué hace el boosting?

1. entrena un modelo inicial,
2. observa en qué casos se equivoca,
3. entrena un nuevo árbol para **corregir esos errores**,
4. repite el proceso muchas veces.

En política pública esto se parece a un programa que cada año corrige sus fallas de focalización:

- primero detecta parte del problema,
- luego aprende de los errores,
- después ajusta la regla,
- y así mejora iterativamente.

### ¿Por qué XGBoost?

Porque es una implementación moderna, eficiente y muy usada en problemas tabulares de alto desempeño.


In [ ]:

# Definimos pipeline con preprocesamiento + XGBoost
xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=250,          # Número de árboles secuenciales
        max_depth=4,               # Profundidad de cada árbol débil
        learning_rate=0.06,        # Tasa de aprendizaje; menor valor = aprendizaje más gradual
        subsample=0.85,            # Muestra de filas por iteración
        colsample_bytree=0.85,     # Muestra de columnas por árbol
        reg_lambda=1.0,            # Regularización L2
        eval_metric="logloss",     # Métrica de optimización
        random_state=42
    ))
])

# Entrenamos el modelo
xgb_model.fit(X_train, y_train)



# 8. Evaluación clásica: Matriz de Confusión y Reporte de Clasificación

En un problema de evasión tributaria, no basta con mirar accuracy.

## Lectura económica de los errores

- **Falso Positivo (FP):** el modelo marca como evasora a una empresa que no lo es.  
  Consecuencia: se desperdicia capacidad de auditoría y puede haber costos reputacionales o administrativos.

- **Falso Negativo (FN):** el modelo deja pasar a una empresa evasora.  
  Consecuencia: pérdida de recaudo y menor efecto disuasorio.

Por eso el criterio no debe ser solo “acertar mucho”, sino **acertar donde importa**.


In [ ]:

# Predicciones duras para Random Forest
rf_pred = rf_model.predict(X_test)

# Predicciones duras para XGBoost
xgb_pred = xgb_model.predict(X_test)

# Probabilidades estimadas para la clase positiva
rf_prob = rf_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# Matrices de confusión
rf_cm = confusion_matrix(y_test, rf_pred)
xgb_cm = confusion_matrix(y_test, xgb_pred)

print("=== Random Forest: Matriz de Confusión ===")
print(rf_cm)

print("\n=== XGBoost: Matriz de Confusión ===")
print(xgb_cm)

print("\n=== Random Forest: Reporte ===")
print(classification_report(y_test, rf_pred))

print("\n=== XGBoost: Reporte ===")
print(classification_report(y_test, xgb_pred))


In [ ]:

# Visualización conjunta de matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Random Forest")
axes[0].set_xlabel("Predicho")
axes[0].set_ylabel("Real")

sns.heatmap(xgb_cm, annot=True, fmt="d", cmap="Greens", ax=axes[1])
axes[1].set_title("XGBoost")
axes[1].set_xlabel("Predicho")
axes[1].set_ylabel("Real")

plt.tight_layout()
plt.show()



# 9. Curva de Ganancia / Lift Curve

Aquí damos el salto desde métricas académicas a **valor económico**.

## Pregunta de negocio público

Si el gobierno solo puede auditar una fracción de empresas,  
¿cuánto mejor es usar el modelo frente a seleccionar empresas al azar?

La **curva de ganancia** responde eso mostrando cuántos evasores acumulamos al auditar primero a los casos con mayor score.

El **lift** compara el rendimiento del modelo con una selección aleatoria.

### Interpretación económica

Si al auditar el 10% más riesgoso detectamos 3 veces más evasores que al azar,  
entonces el modelo multiplica la productividad de la capacidad inspectiva.


In [ ]:

# Construimos un DataFrame de evaluación para XGBoost
# Elegimos XGBoost porque suele ser el candidato más fuerte en tabular data
eval_df = pd.DataFrame({
    "y_true": y_test.values,
    "score": xgb_prob
}).sort_values("score", ascending=False).reset_index(drop=True)

# Proporción acumulada de observaciones auditadas
eval_df["pct_auditado"] = (np.arange(len(eval_df)) + 1) / len(eval_df)

# Evasores detectados acumulados
eval_df["evasores_acumulados"] = eval_df["y_true"].cumsum()

# Porcentaje acumulado de evasores detectados
total_evasores = eval_df["y_true"].sum()
eval_df["ganancia_modelo"] = eval_df["evasores_acumulados"] / total_evasores

# Línea base aleatoria
eval_df["ganancia_aleatoria"] = eval_df["pct_auditado"]

# Lift
eval_df["lift"] = eval_df["ganancia_modelo"] / eval_df["ganancia_aleatoria"]

eval_df.head()


In [ ]:

# Curva de ganancia
plt.plot(eval_df["pct_auditado"], eval_df["ganancia_modelo"], label="Modelo (XGBoost)")
plt.plot(eval_df["pct_auditado"], eval_df["ganancia_aleatoria"], linestyle="--", label="Auditoría al azar")
plt.title("Curva de Ganancia")
plt.xlabel("% acumulado de empresas auditadas")
plt.ylabel("% acumulado de evasores detectados")
plt.legend()
plt.show()


In [ ]:

# Curva de lift
plt.plot(eval_df["pct_auditado"], eval_df["lift"])
plt.axhline(1, linestyle="--", color="red")
plt.title("Lift Curve")
plt.xlabel("% acumulado de empresas auditadas")
plt.ylabel("Lift frente a selección aleatoria")
plt.show()



## Cálculo económico simple de recaudo esperado

Ahora traducimos el ranking a dinero.

Supongamos:

- cada evasor correctamente auditado genera en promedio **\$120.000** de recaudo neto adicional,
- y el Ministerio solo puede auditar el **10%** de las empresas.

Compararemos:
- auditoría guiada por modelo,
- versus auditoría aleatoria.

> Este cálculo es simplificado, pero muy útil para conectar ML con decisión pública.


In [ ]:

# Parámetros del ejercicio económico
porcentaje_auditoria = 0.10
recaudo_promedio_por_caso = 120_000

# Número de empresas auditables
k = int(len(eval_df) * porcentaje_auditoria)

# Evasores detectados por el modelo en el top-k
evasores_modelo_topk = eval_df.head(k)["y_true"].sum()

# Evasores esperados al azar en el mismo top-k
evasores_azar_topk = total_evasores * porcentaje_auditoria

# Recaudo esperado
recaudo_modelo = evasores_modelo_topk * recaudo_promedio_por_caso
recaudo_azar = evasores_azar_topk * recaudo_promedio_por_caso
recaudo_extra = recaudo_modelo - recaudo_azar

print(f"Empresas auditadas: {k}")
print(f"Evasores detectados con modelo: {evasores_modelo_topk}")
print(f"Evasores esperados al azar: {evasores_azar_topk:.1f}")
print(f"Recaudo esperado con modelo: ${recaudo_modelo:,.0f}")
print(f"Recaudo esperado al azar:   ${recaudo_azar:,.0f}")
print(f"Recaudo incremental:        ${recaudo_extra:,.0f}")



# 10. Importancia de variables con SHAP

Las importancias clásicas de árboles ayudan, pero SHAP ofrece una lectura más fina.

## ¿Por qué SHAP?

Porque asigna a cada variable una contribución a la predicción,  
permitiendo responder preguntas como:

- ¿qué empuja a una empresa hacia alto riesgo?
- ¿las multas pesan más que el sector?
- ¿las exenciones funcionan como señal clave?

Esto es especialmente valioso cuando la autoridad tributaria quiere justificar por qué cierto perfil aparece arriba en el ranking de riesgo.

## Cuidado metodológico

SHAP ayuda a **explicar predicciones del modelo**.  
No prueba causalidad.


In [ ]:

# Necesitamos transformar los datos con el preprocesador ya entrenado
X_train_transformed = xgb_model.named_steps["preprocess"].transform(X_train)
X_test_transformed = xgb_model.named_steps["preprocess"].transform(X_test)

# Extraemos nombres de columnas después del One-Hot Encoding
ohe = xgb_model.named_steps["preprocess"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(categorical_features)
feature_names = list(cat_names) + numeric_features

# Construimos DataFrame transformado para SHAP
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

# Extraemos el modelo final entrenado
xgb_final = xgb_model.named_steps["model"]

# Creamos explicador SHAP específico para árboles
explainer = shap.TreeExplainer(xgb_final)

# Calculamos valores SHAP
shap_values = explainer.shap_values(X_test_transformed_df)


In [ ]:

# Resumen global tipo beeswarm
shap.summary_plot(shap_values, X_test_transformed_df, show=False)
plt.title("SHAP Summary Plot - XGBoost")
plt.show()


In [ ]:

# Importancia promedio absoluta por variable
mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = (
    pd.DataFrame({
        "variable": X_test_transformed_df.columns,
        "mean_abs_shap": mean_abs_shap
    })
    .sort_values("mean_abs_shap", ascending=False)
    .head(15)
)

sns.barplot(data=shap_importance, y="variable", x="mean_abs_shap")
plt.title("Top 15 variables según importancia SHAP")
plt.xlabel("Contribución absoluta media")
plt.ylabel("Variable")
plt.show()



# 11. Comparación final entre modelos

En la práctica, un Ministerio no escoge un modelo solo por elegancia técnica.

Debe preguntarse:

1. ¿Cuál detecta mejor evasores?
2. ¿Cuál produce más valor por auditoría?
3. ¿Cuál es más estable?
4. ¿Cuál puedo explicar razonablemente ante control interno, auditoría y ciudadanía?

Resumimos métricas clave para apoyar esa decisión.


In [ ]:

# AUC para ambos modelos
rf_auc = roc_auc_score(y_test, rf_prob)
xgb_auc = roc_auc_score(y_test, xgb_prob)

# DataFrame comparativo
comparison = pd.DataFrame({
    "modelo": ["Random Forest", "XGBoost"],
    "AUC": [rf_auc, xgb_auc],
    "Accuracy": [
        (rf_pred == y_test).mean(),
        (xgb_pred == y_test).mean()
    ]
})

comparison



## Comentario interpretativo

No siempre el modelo “ganador” es el de mayor precisión bruta.

En un entorno público podríamos preferir:

- un modelo ligeramente menos preciso pero más estable e interpretable,
- o un modelo más potente si la herramienta será solo un **sistema de priorización de auditorías**, no una decisión automática final.

La decisión es institucional, no solo algorítmica.



# 12. Taller interactivo — Ejercicios para el estudiante

A continuación vienen tres retos que conectan ciencia de datos, economía pública y MLOps.

---

## Reto 1 — Optimización de hiperparámetros con GridSearchCV

### Objetivo
Buscar mejores combinaciones de:

- `n_estimators`
- `max_depth`

### Pregunta económica
¿La mejora en desempeño justifica el aumento en complejidad computacional y pérdida de interpretabilidad?

> Sugerencia: hágalo primero con Random Forest.


In [ ]:

# ===== RETO 1 =====
# Use GridSearchCV para buscar mejores hiperparámetros para Random Forest.

# Pistas:
# 1. Defina un pipeline con 'preprocess' y RandomForestClassifier().
# 2. Cree un diccionario param_grid con:
#    - model__n_estimators
#    - model__max_depth
# 3. Ejecute GridSearchCV con cv=5 y scoring='roc_auc'.
# 4. Reporte los mejores parámetros y compare contra el modelo base.

# Escriba aquí su solución.



---

## Reto 2 — Evaluar sesgo: ¿el modelo castiga injustamente a las PyMEs?

### Objetivo
Comparar el comportamiento del modelo entre:
- **PyMEs**
- **No PyMEs**

### Pregunta económica y ética
Si el modelo marca a las PyMEs como evasoras con mucha más frecuencia,  
¿eso refleja una señal real de riesgo o un sesgo que penaliza desproporcionadamente a empresas pequeñas?

### Sugerencia
Calcule por grupo:
- tasa de predicción positiva,
- precisión,
- recall,
- falsos positivos.

Luego discuta:
- ¿es justo?
- ¿qué haría usted como analista del Ministerio?


In [ ]:

# ===== RETO 2 =====
# Evalúe si el modelo castiga injustamente a las PyMEs.

# Pistas:
# 1. Tome X_test y y_test.
# 2. Construya un DataFrame con:
#    - y_real
#    - y_pred
#    - es_pyme
# 3. Compare métricas por grupo.
# 4. Discuta si hay sesgo de tratamiento.

# Escriba aquí su solución.



---

## Reto 3 — MLOps: simular Data Drift por recesión

### Escenario
El próximo año ocurre una recesión y los **ingresos brutos de todas las empresas caen 20%**.

### Objetivo
Simular ese nuevo escenario y evaluar si el modelo entrenado con la distribución antigua pierde desempeño.

### Pregunta de MLOps
¿Qué pasa cuando la economía cambia y el modelo sigue siendo el mismo?

Este reto es crucial porque en producción los modelos no fallan solo por mala especificación,  
sino porque **el mundo cambia**.

### Sugerencia
1. Cree una copia de `X_test`.
2. Reduzca `ingresos_brutos` en 20%.
3. Recalcule las *features* derivadas afectadas.
4. Pase el nuevo conjunto por el modelo.
5. Compare métricas con el escenario original.


In [ ]:

# ===== RETO 3 =====
# Simule data drift por recesión: caída de 20% en ingresos.

# Pistas:
# 1. Copie X_test en un nuevo DataFrame.
# 2. Multiplique ingresos_brutos por 0.8.
# 3. Actualice:
#    - ratio_multas_ingresos
#    - ratio_exenciones_ingresos
#    - ingreso_por_empleado
# 4. Prediga con el modelo ya entrenado.
# 5. Compare matriz de confusión / AUC / accuracy.

# Escriba aquí su solución.



# 13. Cierre conceptual

## Lo que debe quedar claro

1. **Random Forest** reduce varianza promediando muchos árboles entrenados sobre muestras bootstrap.
2. **XGBoost** aprende de los errores del modelo anterior y suele rendir muy bien en datos tabulares.
3. En política pública, las métricas deben traducirse en:
   - recaudo,
   - focalización,
   - uso de capacidad,
   - y legitimidad institucional.
4. La interpretabilidad importa, pero también importa el rendimiento cuando la capacidad de intervención es limitada.
5. Un buen modelo en laboratorio puede degradarse en producción si hay **data drift**.

## Mensaje final para economistas

Un modelo de ML no reemplaza el juicio económico.  
Lo complementa.

La pregunta no es “¿el algoritmo decide por nosotros?”  
La pregunta correcta es:

> **¿cómo usamos mejor la predicción para asignar recursos públicos escasos con más evidencia, más eficiencia y más responsabilidad?**
